## 02 Customer Segmentation

This notebook builds customer segments from H&M transaction behavior and customer metadata.

Output: A customer segmentation table.


In [3]:
import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt

#### Load processed data


In [4]:
PROCESSED_DIR = "../data/processed/hm"

customers = pd.read_csv(os.path.join(PROCESSED_DIR, "customers.csv"))
articles = pd.read_csv(os.path.join(PROCESSED_DIR, "articles.csv"))
transactions = pd.read_csv(os.path.join(PROCESSED_DIR, "transactions.csv"))

print(customers.shape)
print(articles.shape)
print(transactions.shape)

(1371980, 8)
(105542, 30)
(500000, 5)


In [5]:
articles.head()

,article_id,product_code,product_name,product_type_no,product_type,product_group,graphical_appearance_no,graphical_appearance,colour_group_code,color_group,...,section_no,section,garment_group_no,garment_group,description,image_path,image_exists,product_purchase_count,unique_customer_count,avg_selling_price
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,../data/raw/hm/images/010/0108775015.jpg,True,175.0,172.0,0.008139
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,../data/raw/hm/images/010/0108775044.jpg,True,116.0,116.0,0.008196
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,../data/raw/hm/images/010/0108775051.jpg,True,2.0,2.0,0.004559
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",../data/raw/hm/images/011/0110065001.jpg,True,19.0,19.0,0.020756
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",../data/raw/hm/images/011/0110065002.jpg,True,11.0,11.0,0.016932


In [6]:
customers.head()

,customer_id,fashion_news_binary,is_active,club_member_status,fashion_news_frequency,age,postal_code,age_group
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,0.0,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,Adult
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0.0,0.0,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...,Young Adult
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.0,0.0,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...,Gen Z
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0.0,0.0,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...,Mature
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...,Mature


In [7]:
transactions.head()

,transaction_date,customer_id,article_id,price,sales_channel_id
0,2019-09-13,215895f90002eb3d1a04bd603513c8e85e6002ef08f136...,786586001,0.022017,1
1,2019-02-23,7b183268e3a4623b80d5325ec4a20a0af0edff7bcb1748...,658911001,0.028797,2
2,2019-07-17,2eb7412239a90c0570cd3d1bf0492856ae5b59058b1ea6...,759326005,0.050831,2
3,2019-05-16,74f162e5a170fd57207aa2a7d5c58479ee9de903b2a277...,737137004,0.027102,1
4,2019-08-10,aab9306ee28c4db494003955f80355e540b01480ab35cf...,785931001,0.050831,2


#### Merge product context into transactions


In [ ]:
txn = transactions.merge(
    articles[
        [
            "article_id",
            "product_type",
            "product_group",
            "color_group",
            "index_group",
            "garment_group",
            "avg_selling_price"
        ]
    ],
    on="article_id",
    how="left"
)

txn.head()

# Build customer behavior features


In [ ]:
snapshot_date = txn["transaction_date"].max() + pd.Timedelta(days=1)

customer_behavior = (
    txn.groupby("customer_id")
    .agg(
        total_transactions=("article_id", "count"),
        unique_products=("article_id", "nunique"),
        total_spend=("price", "sum"),
        avg_price=("price", "mean"),
        max_price=("price", "max"),
        first_purchase_date=("transaction_date", "min"),
        last_purchase_date=("transaction_date", "max")
    )
    .reset_index()
)

# Days since last purchase
customer_behavior["days_since_last_purchase"] = (
    snapshot_date - customer_behavior["last_purchase_date"]
).dt.days

# Customer lifetime in days
customer_behavior["customer_lifetime_days"] = (
    customer_behavior["last_purchase_date"] - customer_behavior["first_purchase_date"]
).dt.days + 1

# Purchase frequency (transactions per day)
customer_behavior["purchase_frequency"] = (
    customer_behavior["total_transactions"] / customer_behavior["customer_lifetime_days"]
)

customer_behavior.head()